# Experiment 2 — RAG-Driven Data Cleaning (New Dataset)
**Query dataset:** Dataset 1 (25 rows sample)  
**Knowledge base:** Datasets 2, 3, 4  
**LLM:** Llama 3.1 8B via Ollama (local)  
**Configs:** LLM-only | RAG-full | RAG-partial (50%)

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from sentence_transformers import SentenceTransformer, util

## 2. Load Datasets

In [ ]:
df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")

# Knowledge base = datasets 2, 3, 4
kb = pd.concat([df2, df3, df4], ignore_index=True)

print(f"Dataset 1 (query): {len(df1)} rows")
print(f"Knowledge base: {len(kb)} rows")
print(f"\nDataset 1 columns:\n{list(df1.columns)}")

## 3. Target Attributes

In [ ]:
TARGET_ATTRIBUTES = [
    "model_number",
    "bus_type",
    "interface_type",
    "form_factor",
    "storage_connection_type"
]

print("Missing values in Dataset 1:")
print(df1[TARGET_ATTRIBUTES].isnull().sum())

## 4. Build Evaluation Set (Ground Truth via cluster_id)

In [ ]:
def get_ground_truth(cluster_id, attribute, kb):
    """Find ground truth value using cluster_id match in knowledge base."""
    matches = kb[kb["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

# Build eval records: rows where attribute is missing AND ground truth is recoverable
eval_records = []
for idx, row in df1.iterrows():
    for attr in TARGET_ATTRIBUTES:
        if pd.isna(row.get(attr)):
            gt = get_ground_truth(row["cluster_id"], attr, kb)
            if gt is not None:
                eval_records.append({
                    "df1_idx": idx,
                    "cluster_id": row["cluster_id"],
                    "attribute": attr,
                    "ground_truth": gt
                })

eval_df = pd.DataFrame(eval_records)
print(f"Full eval set: {len(eval_df)} attribute-level tasks")
print(eval_df["attribute"].value_counts())

## 5. Sample 25 Query Rows

In [ ]:
# Sample 25 unique rows from df1 that appear in the eval set
unique_rows = eval_df["df1_idx"].unique()
sampled_rows = pd.Series(unique_rows).sample(n=min(25, len(unique_rows)), random_state=42).values

# Filter eval_df to only include those 25 rows
eval_df_sample = eval_df[eval_df["df1_idx"].isin(sampled_rows)].reset_index(drop=True)

print(f"Sampled eval set: {len(eval_df_sample)} attribute-level tasks across 25 rows")
print(eval_df_sample["attribute"].value_counts())

## 6. LLM Wrapper (Ollama)

In [ ]:
class OllamaLLMWrapper:
    def __init__(self, model_name="llama3.1:8b",  temperature=0, base_url="http://127.0.0.1:11435"):
        self.model_name = model_name
        self.temperature = temperature
        self.base_url = base_url

    def generate(self, prompt):
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={"model": self.model_name, "prompt": prompt, "stream": False}
        )
        return response.json()["response"]

# Test connection
llm = OllamaLLMWrapper("llama3.1:8b")
test = llm.generate("Say OK")
print("Ollama connection OK:", repr(test[:50]))

## 7. LLM-Only Cleaner

In [ ]:
class LLMOnlyCleaner:
    def __init__(self, llm):
        self.llm = llm

    _EMPTY_SENTINELS = {"none", "nan", "null", "n/a", "na", "unknown", ""}

    def clean_cell(self, row, attribute):
        product = {k: v for k, v in row.items()
                   if k in ["title", "model", "brand", "product_type"]
                   and pd.notna(v)}
        prompt = (
            f"You are a product data expert. Fill in the missing attribute.\n\n"
            f"PRODUCT: {product}\n\n"
            f"MISSING ATTRIBUTE: {attribute}\n\n"
            f"Respond with VALUE:<answer> only. Example: VALUE:PCIe"
        )
        response = self.llm.generate(prompt)
        return self._parse_response(response)

    def _parse_response(self, response):
        for line in response.splitlines():
            line = line.strip()
            if line.upper().startswith("VALUE:"):
                value = line.split(":", 1)[1].strip().strip('"').strip("'")
                if value.lower() not in self._EMPTY_SENTINELS:
                    return value
        # Fallback: short raw answer
        cleaned = response.strip().strip('"').strip("'")
        if cleaned and "\n" not in cleaned and len(cleaned) < 50:
            if cleaned.lower() not in self._EMPTY_SENTINELS:
                return cleaned
        return "UNKNOWN"

## 8. RAG Cleaner

In [ ]:
class RAGCleaner:
    def __init__(self, knowledge_base, llm, top_k=3,
                 retriever_model="sentence-transformers/all-MiniLM-L6-v2"):
        self.kb = knowledge_base.reset_index(drop=True)
        self.llm = llm
        self.top_k = top_k
        self.model = SentenceTransformer(retriever_model)

        print("Encoding knowledge base...")
        self.kb["_text"] = self.kb.apply(self._row_to_text, axis=1)
        self.kb_embeddings = self.model.encode(
            self.kb["_text"].tolist(), convert_to_tensor=True,
            batch_size=64, show_progress_bar=True
        )
        print(f"KB embeddings ready: {self.kb_embeddings.shape}")

    def _row_to_text(self, row):
        attrs = ["title", "model", "model_number", "brand", "product_type"]
        parts = [str(row[a]) for a in attrs if pd.notna(row.get(a))]
        return " | ".join(parts)

    _EMPTY_SENTINELS = {"none", "nan", "null", "n/a", "na", "unknown", ""}

    def _build_prompt(self, row, candidates, attribute):
        product = {k: v for k, v in row.items()
                   if k in ["title", "model", "brand", "product_type"]
                   and pd.notna(row.get(k))}
        key_fields = ["title", "model", "model_number", "bus_type",
                      "interface_type", "form_factor", "storage_connection_type"]
        candidates_text = ""
        for _, c in candidates.iterrows():
            c_dict = {k: v for k, v in c.items()
                      if k in key_fields and pd.notna(c.get(k))}
            candidates_text += f"{c_dict}\n"
        return (
            f"You are a product data expert. Fill in the missing attribute.\n\n"
            f"PRODUCT: {product}\n\n"
            f"SIMILAR REFERENCE PRODUCTS:\n{candidates_text}\n"
            f"MISSING ATTRIBUTE: {attribute}\n\n"
            f"Respond with VALUE:<answer> only. Example: VALUE:PCIe"
        )

    def _parse_response(self, response):
        for line in response.splitlines():
            line = line.strip()
            if line.upper().startswith("VALUE:"):
                value = line.split(":", 1)[1].strip().strip('"').strip("'")
                if value.lower() not in self._EMPTY_SENTINELS:
                    return value
        cleaned = response.strip().strip('"').strip("'")
        if cleaned and "\n" not in cleaned and len(cleaned) < 50:
            if cleaned.lower() not in self._EMPTY_SENTINELS:
                return cleaned
        return "UNKNOWN"

## 9. Evaluation Runner

In [ ]:
def normalize(val):
    return str(val).lower().strip()

def is_correct(predicted, ground_truth):
    p = normalize(predicted)
    g = normalize(ground_truth)
    if p == g:
        return True
    if p in g or g in p:
        return True
    return False

def run_evaluation(eval_df, df1, cleaner, config_name, save_path=None):
    results = []
    total = len(eval_df)
    rows = [df1.loc[task["df1_idx"]] for _, task in eval_df.iterrows()]

    # Precompute query embeddings for RAG configs
    query_embeddings = None
    if isinstance(cleaner, RAGCleaner):
        print(f"Precomputing query embeddings for [{config_name}]...")
        query_texts = [cleaner._row_to_text(row) for row in rows]
        query_embeddings = cleaner.model.encode(
            query_texts, convert_to_tensor=True,
            batch_size=64, show_progress_bar=True
        )
        print("Done.")

    config_start = time.time()

    for i, (_, task) in enumerate(eval_df.iterrows()):
        t0 = time.time()
        row = rows[i]

        if query_embeddings is not None:
            cos_scores = util.cos_sim(query_embeddings[i], cleaner.kb_embeddings)[0]
            top_indices = np.argsort(-cos_scores.cpu().numpy())[:cleaner.top_k]
            candidates = cleaner.kb.iloc[top_indices]
            prompt = cleaner._build_prompt(row, candidates, task["attribute"])
            response = cleaner.llm.generate(prompt)
            predicted = cleaner._parse_response(response)
        else:
            predicted = cleaner.clean_cell(row, task["attribute"])

        gt = task["ground_truth"]
        correct = is_correct(predicted, gt)
        elapsed = time.time() - t0

        results.append({
            "config": config_name,
            "df1_idx": task["df1_idx"],
            "attribute": task["attribute"],
            "ground_truth": gt,
            "predicted": predicted,
            "correct": correct,
            "unknown": predicted == "UNKNOWN"
        })

        if i % 5 == 0:
            elapsed_total = time.time() - config_start
            remaining = (elapsed_total / (i + 1)) * (total - i - 1)
            print(f"[{config_name}] {i+1}/{total} | "
                  f"last: {elapsed:.2f}s | "
                  f"ETA: {remaining/60:.1f} min | "
                  f"attr: {task['attribute']}")

    results_df = pd.DataFrame(results)
    if save_path:
        results_df.to_csv(save_path, index=False)
        print(f"✓ Saved [{config_name}] → {save_path}")
    return results_df

## 10. Run Experiments

In [ ]:
# ── Config 1: LLM-only ────────────────────────────────────────
print("="*50)
print("CONFIG 1: LLM-only")
print("="*50)
results_llm = run_evaluation(
    eval_df_sample, df1,
    LLMOnlyCleaner(llm), "LLM-only",
    save_path="exp2_results_llm_only.csv"
)

In [ ]:
# ── Config 2: RAG full KB ─────────────────────────────────────
print("="*50)
print("CONFIG 2: RAG full KB")
print("="*50)
rag_full = RAGCleaner(knowledge_base=kb, llm=llm, top_k=3)
results_rag_full = run_evaluation(
    eval_df_sample, df1,
    rag_full, "RAG-full",
    save_path="exp2_results_rag_full.csv"
)

In [ ]:
# ── Config 3: RAG partial KB (50%) ───────────────────────────
print("="*50)
print("CONFIG 3: RAG partial KB (50%)")
print("="*50)
kb_partial = kb.sample(frac=0.5, random_state=42).reset_index(drop=True)
rag_partial = RAGCleaner(knowledge_base=kb_partial, llm=llm, top_k=3)
results_rag_partial = run_evaluation(
    eval_df_sample, df1,
    rag_partial, "RAG-partial",
    save_path="exp2_results_rag_partial.csv"
)

## 11. Results

In [ ]:
all_results = pd.concat([results_llm, results_rag_full, results_rag_partial], ignore_index=True)
all_results.to_csv("exp2_results_all.csv", index=False)

print("\n" + "="*50)
print("OVERALL ACCURACY")
print("="*50)
print(all_results.groupby("config")["correct"].mean().round(3))

print("\n" + "="*50)
print("PER-ATTRIBUTE ACCURACY")
print("="*50)
print(all_results.groupby(["config", "attribute"])["correct"].mean().round(3).unstack())

print("\n" + "="*50)
print("UNKNOWN RATE")
print("="*50)
print(all_results.groupby("config")["unknown"].mean().round(3))

## 12. Sample Predictions (Sanity Check)

In [ ]:
# Show a sample of predictions for a quick sanity check
sample = all_results[all_results["attribute"] == "bus_type"].head(15)
print(sample[["config", "ground_truth", "predicted", "correct"]].to_string(index=False))

In [ ]:
print(eval_df_sample["attribute"].value_counts())